# MiniMegatron — GPU verification

Runs every check the project **cannot** run on the developer's machine (Apple Silicon: no
CUDA, no Triton — ISSUES #3). Takes about 10 minutes, most of it installing torch.

**What this closes**

| | |
|---|---|
| ISSUES #16 | the Triton RMSNorm kernel has never been executed |
| DoD #3 | single-GPU training works on CUDA |
| DoD #7 | a Triton kernel passes numerical and gradient tests |
| M5 | kernel runtime and bandwidth vs the PyTorch reference |
| M3 (part) | FP32 vs BF16 — **only on Ampere or newer** |

**What it cannot close.** Colab gives **one GPU per runtime**, so the 1-vs-2-GPU,
DDP-vs-FSDP-vs-TP and peak-memory-per-rank comparisons (DoD #4, #5) are out of reach here.
For those, run the same script on a 2-GPU machine — Kaggle's free `T4 x2`, or a rented
instance — and its multi-GPU section runs automatically.

**Pick the runtime deliberately.** *Runtime → Change runtime type → GPU*. A **T4 has no *native* bf16 (sm_75)
**, so bf16 runs there but *slower* than fp32; the FP32-vs-BF16 speedup needs an L4 or A100.
errors clearly on a T4 rather than silently changing numerics, which the script confirms.

> The kernel tests **passed on their first ever run** on a T4 x2 (2026-11-09);
> finding bugs is the point, and the suite is arranged so they surface as test failures
> rather than as silently wrong training.


## 1. What GPU did we get?


In [ ]:
import torch
print('torch       ', torch.__version__)
print('cuda        ', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print('gpu         ', p.name)
    print('arch        ', f'sm_{p.major}{p.minor}')
    print('memory      ', f'{p.total_memory / 2**30:.1f} GiB')
    print('gpu count   ', torch.cuda.device_count(), '(1 on Colab)')
    print('bf16        ', torch.cuda.is_bf16_supported(), '(False on a T4 -- sm_75)')
else:
    print('\nNo GPU. Runtime -> Change runtime type -> GPU, then rerun this cell.')


## 2. Get the repository

`krishp0130/MiniMegatron` is **private**, so a plain `git clone` will not work. Two options:

- **A fine-grained personal access token** with read access to the repo — the cell below
  prompts for it and does not store it in the notebook.
- **Upload a zip** instead: `git archive -o mm.zip HEAD` locally, then use the commented
  fallback cell.


In [ ]:
import getpass, subprocess

USER = 'krishp0130'
REPO = 'MiniMegatron'
token = getpass.getpass('GitHub token (read access to the repo, not echoed): ')
url = f'https://{token}@github.com/{USER}/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, REPO], check=True)
del token, url  # keep it out of the notebook's memory and output
%cd {REPO}
!git log --oneline -1


In [ ]:
# Fallback, if you would rather not use a token: upload a zip made with
#     git archive -o mm.zip HEAD
# from google.colab import files
# files.upload()
# !mkdir -p MiniMegatron && unzip -q mm.zip -d MiniMegatron
# %cd MiniMegatron


## 3. Install

`torch` is pinned exactly to 2.14.1 (D1), so this replaces Colab's preinstalled build and
downloads a CUDA wheel. **If Colab asks you to restart the session, do it and then re-run
the `%cd` cell above before continuing.** `gpu` is the Triton extra.


In [ ]:
!pip install -q -e '.[dev,gpu]'
!python scripts/check_env.py


## 4. The headline: has the Triton kernel ever worked?

The kernel tests are parameterized over backend, so they judge the kernel by the PyTorch
reference's **own** assertions — 8 shapes including non-powers-of-two, 3 dtypes, extreme
inputs, eps handling. No new test code was written by someone who already knew how the
kernel behaves.


In [ ]:
!python -m pytest tests/kernels -q -rf


## 5. Everything else that needs a GPU

One script: the full suite on GPU, single-GPU training, the kernel benchmark (forward and
backward), FP32 throughput, and FP32-vs-BF16 if the GPU supports bf16. On a 2-GPU machine
it also runs the DDP/FSDP/TP comparisons.


In [ ]:
!bash scripts/gpu_verify.sh


## 6. Collect the results

Copy the tables below into `docs/PROGRESS.md` under **Measured results**, with the GPU name
and the commit. Then update the ISSUES #16 row and the Definition-of-Done table in the
README to match what actually ran.

**Record only what this session produced.** If the kernel tests failed, that is the result —
write it down and fix the kernel, rather than reporting the benchmark that ran anyway.


In [ ]:
!cat results/benchmarks.md 2>/dev/null || echo 'no end-to-end benchmark records'
print()
!python -c "import json;[print(json.dumps({k:v for k,v in json.loads(l).items() if k!='environment'})) for l in open('results/kernels_gpu.jsonl')]" 2>/dev/null || echo 'no kernel records'


In [ ]:
# Download everything for pasting back into the repo.
from google.colab import files
!tar czf gpu_results.tgz results/
files.download('gpu_results.tgz')
